In [ ]:
from __future__ import annotations
import os
from pathlib import Path
from types import SimpleNamespace
import pandas as pd
from tqdm.auto import tqdm
from notebooks.utils.classification_experiment_utils import (
    DATASETS, EVALUATION_REGIMES, TRAINING_REGIMES, build_feature_parent_cohorts,
    discover_model_artifacts, evaluate_feature_vector_model, export_result_frames,
    append_result_frame, export_semeval_result_frames, load_episode_partitions, release_frames,
)
from notebooks.utils.feature_aggregation_utils import FeatureSet, ensure_prepared_partition, load_prepared_partition, prepared_partition_path
from notebooks.utils.local_evidence_evaluation_utils import evaluate_local_evidence_semeval
from notebooks.utils.model_training_utils import load_model_artifact, load_model_metadata
DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
MODELS_ROOT = DATA_ROOT / "experiment-results" / "classification" / "models"
RUN_ID = "feature-vector-classification-2026-09-15-rerun-2"
SCORER_LLMS = ("gemma_3_1b_it", "gemma_3_270m_it", "littlelamb", "smollm2_360m")
SOURCE_SPLITS = {"Kaggle": ("HG", "MG"), "MAGE": ("HG", "MG"), "Outfox": ("HG", "MG"), "BBC": ("HG",)}
EVALUATION_WINDOWS = ("whole_text", 32, 64, 128)
MODEL_FAMILIES = ("logistic_regression",)
SEMEVAL_DATASETS = {"train": ("semeval2024_task8_subtask_c_train", DATA_ROOT / "raw/semevals_2024-task8/SemEval2024-task8/subtaskC/SemEval2024_task8_C_train_MG.csv")}
artifact_paths = discover_model_artifacts(MODELS_ROOT, filters={})
semeval_configurations = {}
for artifact_path in artifact_paths:
    try:
        metadata = load_model_metadata(artifact_path, expected={})
        training_window = str(metadata["training_window"])
        if str(metadata.get("scorer_llm")) in SCORER_LLMS and training_window != "whole_text":
            semeval_configurations[(str(metadata["scorer_llm"]), str(metadata["feature_set"]), int(training_window))] = tuple(metadata["feature_columns"])
    except (FileNotFoundError, KeyError, ValueError, OSError, TypeError):
        continue
semeval_windows, semeval_preparation_errors = {}, {}
for (scorer_llm, feature_set_name, semeval_window), feature_columns in tqdm(semeval_configurations.items(), desc="Preparing SemEval feature partitions", unit="configuration"):
    for semeval_split, (semeval_dataset, semeval_path) in tqdm(SEMEVAL_DATASETS.items(), desc=f"{scorer_llm}/{feature_set_name}/{semeval_window}", unit="split", leave=False):
        try:
            feature_set = FeatureSet(feature_set_name, feature_columns)
            prepared = ensure_prepared_partition(DATA_ROOT, DATA_ROOT / "processed", scorer_llm=scorer_llm, dataset=semeval_dataset, split="MG", feature_set=feature_set, window_configuration=semeval_window, stride=16)
            if prepared["status"] != "available": raise ValueError(str(prepared["error"]))
            expected = {"schema_version": 1, "scorer_llm": scorer_llm, "dataset": semeval_dataset, "split": "MG", "feature_set": feature_set_name, "feature_columns": list(feature_columns), "window_configuration": str(semeval_window), "stride": 16}
            semeval_windows[(scorer_llm, semeval_dataset, feature_set_name, semeval_window)] = load_prepared_partition(prepared["path"], expected=expected).assign(window_length=semeval_window)
        except (FileNotFoundError, KeyError, ValueError, OSError) as error:
            semeval_preparation_errors[(scorer_llm, semeval_dataset, feature_set_name, semeval_window)] = str(error)
exports = export_result_frames(DATA_ROOT, run_id=RUN_ID, frames={name: pd.DataFrame() for name in ("training_metrics", "training_coverage", "evaluation_metrics", "evaluation_predictions", "evaluation_coverage", "run_summary")})
semeval_exports = export_semeval_result_frames(DATA_ROOT, run_id=RUN_ID, frames={name: pd.DataFrame() for name in ("semeval_summary", "semeval_evaluations", "semeval_predicted_regions", "semeval_boundary_summary", "semeval_boundary_predictions", "semeval_boundary_exclusions")})


In [ ]:
evaluation_metrics, evaluation_coverage, run_summary = [], [], []
for artifact_path in tqdm(artifact_paths, desc="Evaluating classifier artifacts", unit="artifact"):
    try:
        metadata = load_model_metadata(artifact_path, expected={})
        scorer_llm, feature_set_name = str(metadata["scorer_llm"]), str(metadata["feature_set"])
        feature_columns, model_family = tuple(metadata["feature_columns"]), str(metadata["model_family"])
        if scorer_llm not in SCORER_LLMS or model_family not in MODEL_FAMILIES: continue
        fitted = load_model_artifact(artifact_path, expected={"model_family": model_family, "scorer_llm": scorer_llm, "feature_set": feature_set_name, "feature_columns": list(feature_columns)}, trust_artifact=True)
    except (FileNotFoundError, KeyError, ValueError, OSError) as error:
        run_summary.append({"run_id": RUN_ID, "stage": "evaluation", "status": "unavailable", "scorer_llm": "", "dataset": "", "split": "", "feature_set": "", "training_window": "", "evaluation_window": "", "model_family": "", "artifact_path": str(artifact_path), "error": str(error)}); append_result_frame(exports["run_summary"], pd.DataFrame([run_summary[-1]]))
        continue
    except Exception:
        raise
    training_regime, training_window = str(metadata["training_regime"]), str(metadata["training_window"])
    if training_window != "whole_text":
      try:
        semeval_window = int(training_window)
        local_classifier = SimpleNamespace(model_name=model_family, feature_columns=feature_columns, estimator=fitted.estimator)
        semeval_model = SimpleNamespace(llm_name=scorer_llm, classifiers={(model_family, semeval_window): local_classifier}, feature_columns=feature_columns, window_lengths=(semeval_window,), stride=int(metadata.get("stride") or 16), top_k=int(metadata.get("top_k", 20)))
        for semeval_split, (semeval_dataset, semeval_path) in SEMEVAL_DATASETS.items():
            try:
                if (scorer_llm, semeval_dataset, feature_set_name, semeval_window) in semeval_preparation_errors: raise ValueError(semeval_preparation_errors[(scorer_llm, semeval_dataset, feature_set_name, semeval_window)])
                prepared_windows = semeval_windows[(scorer_llm, semeval_dataset, feature_set_name, semeval_window)]
                result = evaluate_local_evidence_semeval(semeval_model, DATA_ROOT / "processed", pd.read_csv(semeval_path), dataset=semeval_dataset, split="MG", windows_override=prepared_windows, threshold=0.5, smoothing_window=5, minimum_region_length=3, sustained_length=3)
                dimensions = {"run_id": RUN_ID, "scorer_llm": scorer_llm, "feature_set": feature_set_name, "training_regime": training_regime, "training_window": training_window, "model_family": model_family, "artifact_path": str(artifact_path), "artifact_run_id": metadata["run_id"], "artifact_created_at": metadata["created_at"], "training_datasets": "|".join(metadata["training_datasets"]), "cohort_seed": metadata["cohort_seed"], "stride": metadata["stride"], "train_row_count": metadata["train_row_count"], "validation_row_count": metadata["validation_row_count"], "train_parent_count": metadata["train_parent_count"], "validation_parent_count": metadata["validation_parent_count"], "train_hg_row_count": metadata["train_hg_row_count"], "train_mg_row_count": metadata["train_mg_row_count"], "validation_hg_row_count": metadata["validation_hg_row_count"], "validation_mg_row_count": metadata["validation_mg_row_count"], "evaluation_regime": "SemEval", "evaluation_dataset": semeval_dataset, "semeval_split": semeval_split}
                for name, frame in (("semeval_summary", result.summary), ("semeval_evaluations", result.evaluations), ("semeval_predicted_regions", result.predicted_regions), ("semeval_boundary_summary", result.boundary_summary), ("semeval_boundary_predictions", result.boundary_predictions), ("semeval_boundary_exclusions", result.boundary_exclusions)): append_result_frame(semeval_exports[name], frame.assign(**dimensions))
                release_frames(result.summary, result.evaluations, result.predicted_regions, result.boundary_summary, result.boundary_predictions, result.boundary_exclusions)
                run_summary.append({"run_id": RUN_ID, "stage": "semeval", "status": "available", "scorer_llm": scorer_llm, "dataset": semeval_dataset, "split": "MG", "feature_set": feature_set_name, "training_window": training_window, "evaluation_window": "", "model_family": model_family, "artifact_path": str(artifact_path), "error": ""}); append_result_frame(exports["run_summary"], pd.DataFrame([run_summary[-1]]))
            except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                append_result_frame(semeval_exports["semeval_boundary_exclusions"], pd.DataFrame([{"run_id": RUN_ID, "scorer_llm": scorer_llm, "feature_set": feature_set_name, "training_regime": training_regime, "training_window": training_window, "model_family": model_family, "artifact_path": str(artifact_path), "evaluation_regime": "SemEval", "evaluation_dataset": semeval_dataset, "semeval_split": semeval_split, "reason": str(error)}]))
                run_summary.append({"run_id": RUN_ID, "stage": "semeval", "status": "unavailable", "scorer_llm": scorer_llm, "dataset": semeval_dataset, "split": "MG", "feature_set": feature_set_name, "training_window": training_window, "evaluation_window": "", "model_family": model_family, "artifact_path": str(artifact_path), "error": str(error)}); append_result_frame(exports["run_summary"], pd.DataFrame([run_summary[-1]]))
      except (TypeError, ValueError) as error:
        for semeval_split, (semeval_dataset, _) in SEMEVAL_DATASETS.items(): append_result_frame(semeval_exports["semeval_boundary_exclusions"], pd.DataFrame([{"run_id": RUN_ID, "scorer_llm": scorer_llm, "feature_set": feature_set_name, "training_regime": training_regime, "training_window": training_window, "model_family": model_family, "artifact_path": str(artifact_path), "evaluation_regime": "SemEval", "evaluation_dataset": semeval_dataset, "semeval_split": semeval_split, "reason": str(error)}]))
    for evaluation_regime, source_names in tqdm(EVALUATION_REGIMES.items(), desc="Evaluation regimes", unit="regime", leave=False):
        for evaluation_window in tqdm(EVALUATION_WINDOWS, desc=f"{evaluation_regime} windows", unit="window", leave=False):
            frames, parent_frame = {}, None
            try:
                for source_name in source_names:
                    dataset, split_frames = DATASETS[source_name], []
                    for split in SOURCE_SPLITS[source_name]:
                        path = prepared_partition_path(DATA_ROOT, scorer_llm=scorer_llm, dataset=dataset, split=split, feature_set=feature_set_name, window_configuration=evaluation_window)
                        expected = {"schema_version": 1, "scorer_llm": scorer_llm, "dataset": dataset, "split": split, "feature_set": feature_set_name, "feature_columns": list(feature_columns), "window_configuration": "whole_text" if evaluation_window == "whole_text" else str(evaluation_window), "stride": None if evaluation_window == "whole_text" else 16}
                        split_frames.append(load_prepared_partition(path, expected=expected).assign(dataset=dataset, scorer_llm=scorer_llm, feature_set=feature_set_name))
                    frames[dataset] = pd.concat(split_frames, ignore_index=True)
                parent_frame = pd.concat([frame[["dataset", "split", "text_id", "label"]] for frame in frames.values()], ignore_index=True)
                cohorts = build_feature_parent_cohorts(parent_frame, seed=int(metadata.get("cohort_seed", 42)), max_per_split=int(metadata.get("max_per_split", 500)))
                validation_cohort, test_cohort = cohorts["validation"], cohorts["test"]
                release_frames(parent_frame, *frames.values()); frames = {}
                metadata_for_run = {**metadata, "training_window": training_window, "evaluation_window": "whole_text" if evaluation_window == "whole_text" else str(evaluation_window), "evaluation_regime": evaluation_regime}
                for partition in ("validation", "test"):
                    selected_cohort = validation_cohort if partition == "validation" else test_cohort
                    frames = load_episode_partitions(DATA_ROOT, scorer_llm=scorer_llm, datasets=tuple(DATASETS[name] for name in source_names), partition=partition, feature_set=feature_set_name, window_configuration=evaluation_window, feature_columns=feature_columns, cohort=selected_cohort)
                    metrics, predictions, coverage = evaluate_feature_vector_model(fitted, frames, partition=partition, metadata=metadata_for_run)
                    evaluation_metrics.extend(metrics.to_dict(orient="records")); evaluation_coverage.extend(coverage.to_dict(orient="records")); append_result_frame(exports["evaluation_metrics"], metrics); append_result_frame(exports["evaluation_predictions"], predictions); append_result_frame(exports["evaluation_coverage"], coverage); release_frames(metrics, predictions, coverage)
                    run_summary.append({"run_id": RUN_ID, "stage": "evaluation", "status": "available", "scorer_llm": scorer_llm, "dataset": ", ".join(source_names), "split": partition, "feature_set": feature_set_name, "training_window": training_window, "evaluation_window": metadata_for_run["evaluation_window"], "model_family": model_family, "artifact_path": str(artifact_path), "error": ""}); append_result_frame(exports["run_summary"], pd.DataFrame([run_summary[-1]]))
            except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                run_summary.append({"run_id": RUN_ID, "stage": "evaluation", "status": "unavailable", "scorer_llm": scorer_llm, "dataset": evaluation_regime, "split": "", "feature_set": feature_set_name, "training_window": training_window, "evaluation_window": str(evaluation_window), "model_family": model_family, "artifact_path": str(artifact_path), "error": str(error)}); append_result_frame(exports["run_summary"], pd.DataFrame([run_summary[-1]]))
                continue
            finally: release_frames(parent_frame, *frames.values())
    release_frames(fitted)


In [ ]:
exports
semeval_exports
